In [0]:
%sql

DROP TABLE IF EXISTS db_academy.pii_data.customers_cols_masking;

CREATE OR REPLACE TABLE db_academy.pii_data.customers_cols_masking AS
SELECT customer_id,
        full_name,
        email,
        CAST(phone AS STRING) AS phone,
        ssn,
        CAST(date_of_birth AS STRING) date_of_birth,
        address,
        city,
        country
FROM db_academy.pii_data.customers_raw

In [0]:
%sql
SELECT *
FROM db_academy.pii_data.customers_cols_masking LIMIT 10;

In [0]:
%sql

---------------- MASKING full_name column -----------------------
CREATE OR REPLACE FUNCTION db_academy.pii_data.customer_full_name_masking(full_name STRING)
RETURNS STRING
RETURN 
    CASE WHEN is_account_group_member('admin') THEN full_name
    ELSE concat(left(full_name, 1), repeat('*', length(full_name) - 2)) END;

---------------- MASKING email column -----------------------
CREATE OR REPLACE FUNCTION db_academy.pii_data.customer_email_masking(email STRING)
RETURNS STRING
RETURN 
    CASE WHEN is_account_group_member('admin') THEN email
    ELSE concat(left(email, 1), "******", split(email, '@')[1]) END;

---------------- MASKING phone column -----------------------
CREATE OR REPLACE FUNCTION db_academy.pii_data.customer_phone_masking(phone STRING)
RETURNS STRING
RETURN 
    CASE WHEN is_account_group_member('admin') THEN phone
    ELSE concat('***-***-', right(phone, 4)) END;

---------------- MASKING ssn column -----------------------
CREATE OR REPLACE FUNCTION db_academy.pii_data.customer_ssn_masking(ssn STRING)
RETURNS STRING
RETURN 
    CASE WHEN is_account_group_member('admin') THEN ssn
    ELSE concat('***-**-',right(ssn, 4)) END;

---------------- MASKING date_of_birth column -----------------------
CREATE OR REPLACE FUNCTION db_academy.pii_data.customer_date_of_birth_masking(date_of_birth STRING)
RETURNS STRING
RETURN 
    CASE WHEN is_account_group_member('admin') THEN date_of_birth
    ELSE concat(YEAR(CAST(date_of_birth AS DATE)),'-**-**') END;

---------------- MASKING address column -----------------------
CREATE OR REPLACE FUNCTION db_academy.pii_data.customer_address_masking(address STRING)
RETURNS STRING
RETURN 
    CASE WHEN is_account_group_member('admin') THEN address
    ELSE concat(regexp_extract(address, r"^\d+", 0), " ***************") END;

In [0]:
%sql

SELECT * FROM db_academy.pii_data.customers_cols_masking LIMIT 10;

In [0]:
%sql

ALTER TABLE db_academy.pii_data.customers_cols_masking
ALTER COLUMN full_name SET MASK db_academy.pii_data.customer_full_name_masking;

ALTER TABLE db_academy.pii_data.customers_cols_masking
ALTER COLUMN email SET MASK db_academy.pii_data.customer_email_masking;

ALTER TABLE db_academy.pii_data.customers_cols_masking
ALTER COLUMN phone SET MASK db_academy.pii_data.customer_phone_masking;

ALTER TABLE db_academy.pii_data.customers_cols_masking
ALTER COLUMN ssn SET MASK db_academy.pii_data.customer_ssn_masking;

ALTER TABLE db_academy.pii_data.customers_cols_masking
ALTER COLUMN date_of_birth SET MASK db_academy.pii_data.customer_date_of_birth_masking;

ALTER TABLE db_academy.pii_data.customers_cols_masking
ALTER COLUMN address SET MASK db_academy.pii_data.customer_address_masking;


In [0]:
%sql

SELECT * FROM db_academy.pii_data.customers_cols_masking LIMIT 10;